# 02. Model Training, Hyperparameter Tuning & Model Selection
## Intelligent Healthcare Disease Risk Prediction & Clinical Decision Support System

**Domain:** Healthcare / Supervised Learning (Binary Classification)  
**Dataset:** Pima Indians Diabetes Dataset (`data/diabetes_data.csv`)  
**Objective:** Train, optimize, benchmark, and evaluate candidate supervised algorithms, and select the champion model for clinical deployment based on multi-metric healthcare evaluation.

## 1. Setup & Data Loading

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from training.preprocessing import prepare_data
from training.evaluate import calculate_metrics, format_metrics_table, clinical_tradeoff_summary
from training.model_selection import get_candidate_models, get_hyperparameter_grids, evaluate_all_models, tune_model

data = prepare_data(data_path='../data/diabetes_data.csv', random_state=42)
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
feature_names = data['feature_names']

print('Data successfully prepared.')
print(f'Training samples (post-SMOTE): {X_train.shape[0]}')
print(f'Testing samples (unaltered):   {X_test.shape[0]}')
print(f'Total features:               {len(feature_names)}')


Data successfully prepared.
Training samples (post-SMOTE): 800
Testing samples (unaltered):   154
Total features:               16


## 2. Supervised Algorithms Benchmarking (6 Models)

In [2]:
comp_df, trained_models, baseline_results = evaluate_all_models(X_train, y_train, X_test, y_test, random_state=42)
cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'ROC-AUC', 'FP', 'FN']
print(comp_df[cols].to_string(index=False))

              Model  Accuracy  Precision  Recall  Specificity  F1-Score  ROC-AUC  FP  FN
Logistic Regression    0.7532     0.6250  0.7407         0.76    0.6780   0.8285  24  14
      Random Forest    0.7597     0.6349  0.7407         0.77    0.6838   0.8214  23  14
  Gradient Boosting    0.7597     0.6349  0.7407         0.77    0.6838   0.8196  23  14
            XGBoost    0.7532     0.6429  0.6667         0.80    0.6545   0.8126  20  18
                SVM    0.7013     0.5645  0.6481         0.73    0.6034   0.7889  27  19
      Decision Tree    0.7208     0.5932  0.6481         0.76    0.6195   0.7041  24  19


## 3. Stratified 5-Fold Cross-Validation Hyperparameter Tuning

In [3]:
grids = get_hyperparameter_grids()
base_models = get_candidate_models(random_state=42)
tuned_results = {}
tuned_models = {}

for model_name in ['Random Forest', 'Gradient Boosting', 'XGBoost']:
    cv_search = tune_model(model_name, base_models[model_name], grids[model_name], X_train, y_train, cv_folds=5, scoring='roc_auc')
    best_est = cv_search.best_estimator_
    tuned_models[f'Tuned {model_name}'] = best_est
    y_pred = best_est.predict(X_test)
    y_prob = best_est.predict_proba(X_test)[:, 1]
    tuned_results[f'Tuned {model_name}'] = calculate_metrics(y_test, y_pred, y_prob)
    print(f'Tuned {model_name} - Best CV ROC-AUC: {cv_search.best_score_:.4f}')

Tuned Random Forest - Best CV ROC-AUC: 0.9004
Tuned Gradient Boosting - Best CV ROC-AUC: 0.8958
Tuned XGBoost - Best CV ROC-AUC: 0.8914


## 4. Final Comparison Leaderboard

In [4]:
all_results = {**baseline_results, **tuned_results}
leaderboard = format_metrics_table(all_results)
print(leaderboard[cols].to_string(index=False))

                  Model  Accuracy  Precision  Recall  Specificity  F1-Score  ROC-AUC  FP  FN
    Logistic Regression    0.7532     0.6250  0.7407         0.76    0.6780   0.8285  24  14
    Tuned Random Forest    0.7532     0.6333  0.7037         0.78    0.6667   0.8273  22  16
          Random Forest    0.7597     0.6349  0.7407         0.77    0.6838   0.8214  23  14
      Gradient Boosting    0.7597     0.6349  0.7407         0.77    0.6838   0.8196  23  14
Tuned Gradient Boosting    0.7403     0.6167  0.6852         0.77    0.6491   0.8167  23  17
                XGBoost    0.7532     0.6429  0.6667         0.80    0.6545   0.8126  20  18
          Tuned XGBoost    0.7403     0.6167  0.6852         0.77    0.6491   0.8093  23  17
                    SVM    0.7013     0.5645  0.6481         0.73    0.6034   0.7889  27  19
          Decision Tree    0.7208     0.5932  0.6481         0.76    0.6195   0.7041  24  19


## 5. Clinical Trade-off Discussion: False Positives vs False Negatives

### Clinical Rationale for Viva Examination:
- **False Negatives (Critical Failure Mode):** Diabetic individuals falsely classified as healthy. Leads to untreated progressive microvascular and macrovascular complications. In medical screening, **Recall / Sensitivity** must be prioritized.
- **False Positives (Manageable Operational Burden):** Healthy patients classified as at-risk. Leads to benign follow-up diagnostic tests (HbA1c / OGTT) and lifestyle guidance, with no physiological risk to the patient.
- **Selection:** **Logistic Regression** and **Random Forest** achieved the lowest False Negatives ($FN = 14$) and highest Sensitivity ($74.07\%$), while Logistic Regression delivered the peak ROC-AUC ($0.8285$).